# Meta-Analysis of Research Papers on Large Language Models (LLMs)

**Week 10, Day 5 Mini Project — Intro to AI Bootcamp**

**Name:** Natanel Karp &nbsp;&nbsp;|&nbsp;&nbsp; **Date:** May 2026

## 1. Introduction

Large Language Models (LLMs) are a type of deep learning model trained on massive amounts of text to predict and generate language. They've become one of the most active areas in AI research, and the progress over the last few years has been fast — from GPT-3 handling decent zero-shot tasks in 2020 to models like GPT-4 and Claude managing complex conversations, reasoning, and code generation by 2023.

The thing is, a model that can technically predict text well isn't the same as a model that's actually useful or safe to deploy. That gap between raw capability and real-world use is what connects the five papers I chose for this analysis. The shared theme is: **how do you make LLMs more aligned with what users want, safer to use, and cheaper to fine-tune and run?** These questions come up across instruction tuning, alignment research, and efficiency work — and each paper approaches it from a different angle.

**Papers included in this analysis:**

1. Ouyang et al. (2022). *Training language models to follow instructions with human feedback.* NeurIPS 2022.
2. Touvron et al. (2023). *Llama 2: Open Foundation and Fine-Tuned Chat Models.* Meta AI / arXiv:2307.09288.
3. Hu et al. (2022). *LoRA: Low-Rank Adaptation of Large Language Models.* ICLR 2022.
4. Bai et al. (2022). *Constitutional AI: Harmlessness from AI Feedback.* Anthropic / arXiv:2212.08073.
5. Jiang et al. (2023). *Mistral 7B.* Mistral AI / arXiv:2310.06825.

## 2. Paper Summaries

### Paper 1 — InstructGPT: Training LMs to Follow Instructions with Human Feedback

**Research Problem:** GPT-3 was trained to predict the next token in a sequence — not to respond helpfully to instructions. This mismatch meant the model would often produce outputs that were off-topic, verbose, or harmful even when given a clear prompt. It's capable of generating fluent text, but that doesn't mean it actually does what you ask.

**Proposed Solution:** The paper introduces RLHF (Reinforcement Learning from Human Feedback) as a three-step pipeline:
1. Supervised fine-tuning on human-written demonstrations
2. Training a reward model from human preference rankings between outputs
3. Using PPO to fine-tune the LM to maximise the reward model's score

The resulting model is called **InstructGPT**.

**Main Results:** InstructGPT at 1.3B parameters was preferred over GPT-3 at 175B in human evaluations — a model 100x smaller outperforming the base model purely through alignment training. It also produced significantly fewer toxic and hallucinated outputs on TruthfulQA and RealToxicityPrompts.

**Datasets, Architecture, Metrics:** Built on GPT-3. Training data sampled from real OpenAI API prompts plus human-written examples. Evaluated through human preference ratings, TruthfulQA, and RealToxicityPrompts.

### Paper 2 — Llama 2: Open Foundation and Fine-Tuned Chat Models

**Research Problem:** Most of the best-performing LLMs at the time — GPT-4, Claude — were closed source. Researchers couldn't inspect the weights, study the architecture, or build on them directly. There was a clear need for a high-quality, open chat model that also took safety seriously.

**Proposed Solution:** Meta released Llama 2, a family of pre-trained and fine-tuned models at 7B, 13B, and 70B scales. The Llama 2-Chat variants use a similar RLHF pipeline to InstructGPT plus a technique called **Ghost Attention (GAtt)**, which helps the model maintain consistent system-level instructions across long multi-turn conversations without drifting.

**Main Results:** Llama 2-Chat (70B) is competitive with GPT-3.5 on helpfulness benchmarks. The 13B version outperforms most other open-source models at the 30B+ scale. Safety evaluations also put it above many open and closed models at the time.

**Datasets, Architecture, Metrics:** Pre-trained on 2 trillion tokens. Decoder-only Transformer with grouped-query attention (GQA). Evaluated on MMLU, HumanEval, GSM8K, MT-Bench, and human preference studies.

### Paper 3 — LoRA: Low-Rank Adaptation of Large Language Models

**Research Problem:** Full fine-tuning of large models means updating every single parameter — for GPT-3 that's 175 billion of them. That's computationally expensive, memory-intensive, and you need a separate full model copy for every task you fine-tune on. This makes adapting LLMs to specific use cases inaccessible for most researchers and smaller teams.

**Proposed Solution:** LoRA freezes all the pre-trained model weights and injects trainable low-rank decomposition matrices into each Transformer layer. For a weight matrix **W** of shape `d × d`, rather than updating W directly, LoRA learns two smaller matrices **A** (`d × r`) and **B** (`r × d`) where `r << d`. Only A and B get trained during fine-tuning. At inference, the product BA gets merged back into W — so there's no added latency.

**Main Results:** LoRA matches or beats full fine-tuning on standard NLG benchmarks (E2E, WebNLG, DART for GPT-3; GLUE for RoBERTa/DeBERTa) while:
- Reducing trainable parameters by up to **10,000x**
- Reducing GPU memory usage by around **3x**

**Datasets, Architecture, Metrics:** Tested on GPT-2, GPT-3, RoBERTa, and DeBERTa. Benchmarks include GLUE, E2E NLG, WebNLG, DART, and commonsense reasoning tasks.

### Paper 4 — Constitutional AI: Harmlessness from AI Feedback

**Research Problem:** RLHF requires a lot of human-labeled preference data, which is slow and expensive to collect at scale. There's also no explicit, auditable definition of what "harmless" or "helpful" means — human raters use implicit judgments that are hard to inspect or scale up as models get more capable.

**Proposed Solution:** Constitutional AI uses a written set of explicit principles (a "constitution") to guide model behavior. The process runs in two stages:
1. **SL-CAI:** The model critiques and revises its own potentially harmful responses using the constitution
2. **RLAIF:** AI-generated preference labels replace expensive human labels to train a reward model, which is then used in standard RLHF

**Main Results:** CAI-trained models (early Claude versions) score higher on both helpfulness and harmlessness than RLHF-only models in Anthropic's internal evaluations. The model can also explain why it's declining a request, which is useful for user trust and transparency.

**Datasets, Architecture, Metrics:** Uses Anthropic's internal helpfulness and harmlessness dataset. Decoder-only LM (architecture not disclosed at the time). Evaluated with human preference ratings and automated red-team metrics.

### Paper 5 — Mistral 7B

**Research Problem:** A common assumption across LLM research is that more parameters equals better performance. But large models are expensive to train and serve. Mistral 7B asks whether a smaller model can compete with much larger ones if the architecture and training choices are done carefully enough.

**Proposed Solution:** Mistral 7B introduces two architectural changes:
- **Grouped-Query Attention (GQA):** Reduces inference memory bandwidth and speeds up decoding
- **Sliding Window Attention (SWA):** Allows the model to handle longer contexts without quadratic memory scaling

It was trained on a large, carefully filtered web corpus and released under Apache 2.0.

**Main Results:** Mistral 7B outperforms Llama 2 13B on every benchmark tested and matches Llama 2 34B on most tasks. On reasoning, math, and coding it beats Llama 1 34B — a model roughly five times its size.

**Datasets, Architecture, Metrics:** Trained on a filtered web crawl. Evaluated on HellaSwag, PIQA, WinoGrande, ARC-Easy, ARC-Challenge, BoolQ, MBPP, HumanEval, and GSM8K.

## 3. Comparative Analysis

In [ ]:
# Install pandas for table display (pre-installed in Colab)
import pandas as pd
from IPython.display import display

pd.set_option('display.max_colwidth', None)

In [ ]:
# Table 1: Objectives and Problem Domains
objectives = pd.DataFrame({
    'Paper': ['InstructGPT', 'Llama 2', 'LoRA', 'Constitutional AI', 'Mistral 7B'],
    'Main Focus': ['Alignment / RLHF', 'Open-source + Safety', 'Parameter Efficiency',
                   'Scalable Alignment', 'Architecture Efficiency'],
    'Core Problem': [
        'Making models follow instructions',
        'Democratizing safe, capable LLMs',
        'Reducing fine-tuning cost and memory',
        'Safety without expensive human labeling',
        'High performance at small model scale'
    ]
})

print("=== Objectives and Problem Domains ===")
display(objectives.set_index('Paper'))

In [ ]:
# Table 2: Model Architectures and Key Innovations
architectures = pd.DataFrame({
    'Paper': ['InstructGPT', 'Llama 2', 'LoRA', 'Constitutional AI', 'Mistral 7B'],
    'Architecture': [
        'GPT-3 (175B)',
        'Decoder-only, GQA',
        'Any Transformer',
        'Decoder-only LM',
        'Decoder-only, GQA + SWA'
    ],
    'Key Innovation': [
        'RLHF pipeline (SFT + reward model + PPO)',
        'Ghost Attention for multi-turn consistency',
        'Low-rank adapters injected into frozen weights',
        'AI self-critique + RLAIF pipeline',
        'Sliding window attention for long-context efficiency'
    ]
})

print("=== Model Architectures and Key Innovations ===")
display(architectures.set_index('Paper'))

In [ ]:
# Table 3: Training Strategies
training = pd.DataFrame({
    'Paper': ['InstructGPT', 'Llama 2', 'LoRA', 'Constitutional AI', 'Mistral 7B'],
    'Strategy': [
        'SFT -> Reward Model -> PPO',
        'Pre-training + SFT + RLHF',
        'Frozen base + low-rank adapters',
        'SL-CAI -> RLAIF -> RLHF',
        'Standard pre-training'
    ],
    'Scale': ['1.3B-175B', '7B-70B', 'Any', 'Not disclosed', '7B']
})

print("=== Training Strategies ===")
display(training.set_index('Paper'))

In [ ]:
# Table 4: Strengths, Limitations, and Reproducibility
evaluation = pd.DataFrame({
    'Paper': ['InstructGPT', 'Llama 2', 'LoRA', 'Constitutional AI', 'Mistral 7B'],
    'Strength': [
        'Very influential, strong alignment results',
        'Open weights, strong safety evaluation',
        'Massive cut in fine-tuning cost',
        'Principled, more scalable alignment approach',
        'Strong performance far above its parameter count'
    ],
    'Limitation': [
        'No model release, closed-source base',
        'Needs significant compute at 70B',
        'Minor gaps vs full fine-tune on some tasks',
        'Base model closed, internal data only',
        'Pre-training data not fully documented'
    ],
    'Reproducibility': ['Low', 'Good', 'Excellent', 'Low', 'Good']
})

print("=== Strengths, Limitations, and Reproducibility ===")
display(evaluation.set_index('Paper'))

### Benchmarks and Evaluation

One thing that made comparing these papers tricky is that the evaluation setups are all over the place. InstructGPT and Constitutional AI rely mostly on human preference ratings, which are more meaningful for measuring real-world helpfulness but are expensive to replicate. Llama 2, LoRA, and Mistral 7B use standard benchmarks like MMLU, HumanEval, and GLUE — reproducible, but they don't always capture how a model behaves in open-ended real-world use. There's no universal standard for evaluating LLMs yet, and that's a genuine problem for making cross-paper comparisons.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Visualise reproducibility scores across papers
papers = ['InstructGPT', 'Llama 2', 'LoRA', 'Constitutional AI', 'Mistral 7B']
repro_scores = [1, 3, 5, 1, 3]  # Low=1, Good=3, Excellent=5
colors = ['#e74c3c', '#f39c12', '#2ecc71', '#e74c3c', '#f39c12']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(papers, repro_scores, color=colors, edgecolor='black', linewidth=0.5)
ax.set_xlim(0, 6)
ax.set_xticks([1, 3, 5])
ax.set_xticklabels(['Low', 'Good', 'Excellent'])
ax.set_xlabel('Reproducibility')
ax.set_title('Reproducibility by Paper', fontsize=13, fontweight='bold')

legend_patches = [
    mpatches.Patch(color='#e74c3c', label='Low'),
    mpatches.Patch(color='#f39c12', label='Good'),
    mpatches.Patch(color='#2ecc71', label='Excellent')
]
ax.legend(handles=legend_patches, loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Visualise model scale across papers
fig, ax = plt.subplots(figsize=(10, 5))

paper_labels = ['InstructGPT\n(max)', 'Llama 2\n(max)', 'LoRA\n(GPT-3 base)', 'CAI\n(undisclosed)', 'Mistral 7B']
param_counts = [175, 70, 175, 0, 7]  # billions; CAI=0 as undisclosed
bar_colors = ['#3498db', '#9b59b6', '#1abc9c', '#95a5a6', '#e67e22']

bars = ax.bar(paper_labels, param_counts, color=bar_colors, edgecolor='black', linewidth=0.5)

for bar, val in zip(bars, param_counts):
    label = f'{val}B' if val > 0 else 'N/A'
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            label, ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel('Parameters (Billions)')
ax.set_title('Model Scale Comparison (Max Parameters per Paper)', fontsize=13, fontweight='bold')
ax.set_ylim(0, 210)
plt.tight_layout()
plt.show()

## 4. Insights and Reflection

Something that comes up across all five papers is the move away from "just train a bigger model." InstructGPT showed that a 1.3B aligned model outperforms a 175B base model in practice. LoRA showed you don't need to update every parameter to get strong fine-tuning results. Mistral 7B showed a 7B model with better architecture choices can match a 34B one. It seems like **training strategy, data quality, and architecture matter more than just adding parameters.**

Safety and alignment shows up across all five papers, just in different forms. InstructGPT and Llama 2 both use human-feedback RLHF. Constitutional AI tries to reduce dependence on expensive human labels by using AI-generated feedback instead. The CAI approach seems especially worth watching — if RLAIF scales well, it could make alignment much more accessible to teams that can't afford large human labeling pipelines.

The open-source shift also stood out. Llama 2, Mistral 7B, and LoRA all have public weights or code, and that's clearly sped up downstream research. The gap between open and closed models closed faster than a lot of people expected, and it seems directly tied to the fact that researchers can actually inspect, fine-tune, and build on these models rather than just calling an API.

**Shared limitations across papers:** evaluation is inconsistent and not standardized across the field; pre-training data is often undisclosed or poorly documented; and alignment techniques (RLHF, CAI) don't come with guarantees that they generalize as models scale further. Hallucination and bias are acknowledged in all five papers but none offer a full solution.

**Future directions worth thinking about:**
- Better standardized evaluation covering real-world use rather than just benchmark scores
- Combining LoRA-style efficiency with alignment fine-tuning to lower the cost of safety training
- Scaling RLAIF
- Extending these techniques to multi-modal inputs — all five papers are text-only, which is already a limitation given where the field is heading

## 5. Conclusion

Reading all five of these papers, the "just make it bigger" phase of LLM research seems to be over — or at least seriously challenged. The current focus is on making models genuinely useful through alignment, efficient to adapt through parameter-efficient fine-tuning, and practical for smaller teams through open weights and smaller architectures. RLHF has become the standard alignment tool, but Constitutional AI points toward something more scalable. LoRA has fundamentally changed what individual researchers can do with large models. Mistral 7B proved architecture improvements alone can close a huge performance gap.

What I found most interesting doing this analysis is how interconnected these problems are. Making fine-tuning cheaper (LoRA) also makes safety fine-tuning cheaper. Open weights (Llama 2, Mistral) accelerate alignment research. The next challenges — standardizing evaluation, scaling alignment, handling multi-modal inputs — all seem tractable given the pace of the last few years.

## References

1. Ouyang, L., et al. (2022). *Training language models to follow instructions with human feedback.* NeurIPS 2022. https://arxiv.org/abs/2203.02155

2. Touvron, H., et al. (2023). *Llama 2: Open foundation and fine-tuned chat models.* arXiv:2307.09288. https://arxiv.org/abs/2307.09288

3. Hu, E. J., et al. (2022). *LoRA: Low-rank adaptation of large language models.* ICLR 2022. https://arxiv.org/abs/2106.09685

4. Bai, Y., et al. (2022). *Constitutional AI: Harmlessness from AI feedback.* arXiv:2212.08073. https://arxiv.org/abs/2212.08073

5. Jiang, A. Q., et al. (2023). *Mistral 7B.* arXiv:2310.06825. https://arxiv.org/abs/2310.06825